# L16 — Tracing, lineage and replay-with-fixed-retrieval
**Objectives 13, 16.**

**Northfield Grocers context:** a store manager reports the assistant told an associate the chilled refund window was 48 hours. No error was logged. The platform team must reconstruct exactly what the system knew (index version, chunks, prompt version), replay the request with retrieval fixed to decide whether retrieval or the model was at fault, and — because chunks carry lineage — delete only the affected vectors, not rebuild the index.

**Retail use cases:** incident forensics on a wrong SOP answer; proving to compliance which policy version an answer used; isolating the blast radius of a bad ingestion run.

**Platform:** any (fully executable in SMOKE mode on the provided trace log). On the cluster the trace store is LangSmith / MLflow tracing / OpenTelemetry → Delta.

**Done means:** the failing requests are found by segmenting on prompt version; replay with fixed retrieval attributes the failure to the model/prompt layer; lineage identifies exactly the chunks from the bad ingestion and nothing else.

## Step 1 — Load the trace log and segment failures
*Why:* every request already carries request_id, prompt_version, index_version, retrieved chunk ids, answer, latency, tokens and feedback. Segment by version before guessing.

In [ ]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re, subprocess, importlib, importlib.util
import numpy as np, pandas as pd

def ensure_packages(pkgs):
    """Install any missing pip packages into THIS Python (same mechanism as %pip on Databricks) and import them.
    Fresh packages are importable immediately — no restart. Only labs that need extras call this (L02, L08)."""
    missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
    if not missing:
        print("Packages present:", pkgs); return
    print("Installing missing packages into", sys.executable, ":", missing)
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)   # local system Pythons
    if r.returncode != 0:
        raise ImportError("pip could not install " + str(missing) + ". Ask the admin to add them as cluster libraries "
                          "(Compute → Libraries → PyPI) or use an internal index. pip said: " + r.stderr[-600:])
    importlib.invalidate_caches()
    for p in missing: importlib.import_module(p.replace("-", "_"))
    print("Installed and imported:", missing)

# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")

# Data folder: env override → package-relative (../../data) → Unity Catalog volume → search the workspace once
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    import glob
    _hits = [h for root in ("/Workspace", "/Volumes", os.path.expanduser("~")) if os.path.isdir(root)
             for h in glob.glob(os.path.join(root, "**", "catalog_items.csv"), recursive=True)][:1]
    DATA_DIR = os.path.dirname(_hits[0]) if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Upload the package's data/ folder to a Unity Catalog volume and set "
                            "os.environ['DATA_DIR'] = '/Volumes/<catalog>/<schema>/<volume>' in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}  python={sys.version.split()[0]}")

In [ ]:
traces = pd.DataFrame([json.loads(l) for l in open(os.path.join(DATA_DIR, "trace_log.jsonl"))])
seg = traces.groupby("prompt_version").agg(requests=("request_id", "count"), unknown_rate=("answer", lambda s: (s == "unknown").mean()), feedback=("user_feedback", "mean"), p95_ms=("latency_ms", lambda s: np.percentile(s, 95))).round(3)
print(seg.to_string())
check(set(seg.index) == {"qa-v3", "qa-v4"}, "traces segmented by prompt version")

## Step 2 — Reconstruct one failed request
*Why:* forensics starts from a single request id: what was retrieved, from which index, with which prompt.

In [ ]:
sops = pd.read_csv(os.path.join(DATA_DIR, "sop_chunks.csv")).set_index("chunk_id")
def reconstruct(request_id):
    t = traces.set_index("request_id").loc[request_id]
    chunks = [dict(chunk_id=c, version=sops.loc[c, "version"], text=sops.loc[c, "text"]) for c in t.retrieved_chunk_ids]
    return dict(question=t.question, chunks=chunks, prompt_version=t.prompt_version, index_version=t.index_version, answer=t.answer)
failed = traces[(traces.answer == "unknown")].request_id.iloc[0]
rec = reconstruct(failed); print(json.dumps(rec, indent=1)[:500])
check(rec["chunks"] and rec["index_version"] == "idx-2026-09", "request reconstructed with chunks and versions")

## Step 3 — Replay with fixed retrieval
*Why:* the definitive RAG debugging move. Bypass search, hand the model the exact chunks from the trace, ask the same question. Correct now → retrieval fetched the wrong context. Still wrong → prompt/model. In SMOKE mode the 'model' is the deterministic answerer from L15 (v3 grounded, v4 not).

In [ ]:
def model_answer(question, chunks, prompt_version):
    """Stand-in model: v3 extracts from the first chunk; v4 ignores context 30% of the time (planted)."""
    if prompt_version == "qa-v3": return chunks[0]["text"]
    return "unknown" if hash(question) % 10 < 3 else chunks[0]["text"]
def replay(request_id, prompt_version=None):
    r = reconstruct(request_id); pv = prompt_version or r["prompt_version"]
    new = model_answer(r["question"], r["chunks"], pv)
    correct_chunks = any(r["question"].split()[-2].lower() in c["text"].lower() or True for c in r["chunks"])  # chunks are from the right area by construction
    verdict = ("model/prompt" if new == "unknown" else "retrieval" if not correct_chunks else "passes on replay → transient / model nondeterminism")
    return dict(request_id=request_id, replay_answer=new, verdict=verdict, prompt_version=pv)
fails_v4 = traces[(traces.answer == "unknown") & (traces.prompt_version == "qa-v4")].request_id.head(20)
verdicts = pd.Series([replay(r)["verdict"] for r in fails_v4]).value_counts(); print(verdicts)
v3_replay = pd.Series([replay(r, "qa-v3")["replay_answer"] != "unknown" for r in fails_v4]).mean()
print(f"same requests replayed under qa-v3 answer correctly: {v3_replay:.0%}")
check(v3_replay == 1.0, "replay under the previous prompt version fixes the failures → fault is the prompt, not retrieval")

## Step 4 — Lineage: isolate a bad ingestion run
*Why:* chunks carry chunk_id, version and effective date (and, in production, a source-document hash and ETL run id). When ingestion run 2025-01 is found to have loaded the stale refund rule, delete exactly its vectors.

In [ ]:
sops["etl_run"] = np.where(sops.effective == "2025-01-01", "etl-2025-01", "etl-2026-07")
def blast_radius(etl_run):
    bad = set(sops.index[sops.etl_run == etl_run]); hit = traces[traces.retrieved_chunk_ids.apply(lambda ids: bool(set(ids) & bad))]
    return dict(chunks=sorted(bad), requests_affected=len(hit))
br = blast_radius("etl-2025-01"); print(br)
check(br["chunks"] == ["SOP-004-old"], "lineage isolates exactly the stale chunk from the bad ETL run")

## Step 5 — The incident record
*Why:* what the system knew, what was replayed, what was deleted, what changed. This is the artefact compliance asks for.

In [ ]:
incident = dict(request_id=failed, reconstruction=rec, replay=replay(failed), lineage=br, action="roll back to qa-v3; delete etl-2025-01 vectors; add failed questions to golden set", date=time.strftime("%Y-%m-%d"))
os.makedirs("/tmp/l16", exist_ok=True); json.dump(incident, open("/tmp/l16/incident.json", "w"), indent=2, default=str)
check(os.path.exists("/tmp/l16/incident.json"), "incident record written"); print("L16 complete.")